In [0]:
%pip install yfinance

In [0]:
import yfinance as yf
import pandas as pd
from pyspark.sql import functions as F

In [0]:
# --- Configuration ---
CATALOG = "iran_israel_capstone_project"
SCHEMA = "bronze"
LANDING_PATH = "abfss://capstonecontainer@iranisrael65.dfs.core.windows.net/landing_zone/market_data/"

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

In [0]:
tickers = {
    "^NSEI": "NIFTY_50", "^BSESN": "SENSEX", "^NSEBANK": "NIFTY_BANK",
    "^CNXENERGY": "NIFTY_ENERGY", "^CNXAUTO": "NIFTY_AUTO", "^CNXIT": "NIFTY_IT",
    "HAL.NS": "HAL", "INDIGO.NS": "INDIGO", "ASIANPAINT.NS": "ASIAN_PAINTS",
    "ONGC.NS": "ONGC", "BZ=F": "BRENT_CRUDE", "INR=X": "USD_INR",
    "GC=F": "GOLD", "^INDIAVIX": "INDIA_VIX"
}

In [0]:
# --- 1. API to ADLS Landing Zone ---
print("Fetching data from yfinance API...")
dfs = []
for ticker, name in tickers.items():
    df = yf.download(ticker, start="2023-10-01", end="2025-04-01", interval="1d", progress=False)
    
    # Flatten MultiIndex if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    df = df.reset_index()
    df["ticker"] = ticker
    df["asset_name"] = name
    dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)
market_data_landing = spark.createDataFrame(final_df)

In [0]:
# Write raw parquet to landing zone
market_data_landing.write.mode("overwrite").parquet(LANDING_PATH)
print(f"Data landed successfully in: {LANDING_PATH}")

In [0]:
# --- 2. ADLS Landing Zone to Unity Catalog (Bronze) ---
print("Promoting Landing Data to Bronze Delta Table...")
df_market_raw = spark.read.parquet(LANDING_PATH)

In [0]:
df_market_bronze = df_market_raw.select(
    F.col("Date").alias("trade_date"),
    F.col("Open").alias("open"),
    F.col("High").alias("high"),
    F.col("Low").alias("low"),
    F.col("Close").alias("close"),
    F.col("Volume").alias("volume"),
    "ticker",
    "asset_name"
).withColumn("ingestion_timestamp", F.current_timestamp()) \
 .withColumn("source_file", F.lit("yfinance_market_data"))

In [0]:
# Save to Unity Catalog
df_market_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("market_data")
print("Successfully written to bronze.market_data")